# Part 3 — Multimodal multi-agent system
Planner → Vision → Prompt Engineering → Image Generation → Critique (≤ 2 revisions). Run on a Colab T4: Runtime → Restart & Run All.

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

## Setup
Colab secret needed: `ANTHROPIC_API_KEY`. When asked, upload `repo.bundle` (made with `git bundle create repo.bundle zoheb`). `P3_LORA` is the Part 2 final LoRA on Hugging Face.

In [ ]:
import os
BRANCH = "zoheb"                                            # switch to main once merged
P3_LORA = "sakotiya/data266-team32-naruto-run2-lora-r32"    # Part 2 final LoRA (HF repo id or local folder)
if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    if not os.path.isdir("data266-lab2-team32"):
        from google.colab import files                      # no GitHub token: upload repo.bundle (git bundle create repo.bundle zoheb)
        files.upload()
        !git clone -q -b {BRANCH} repo.bundle data266-lab2-team32
    %cd data266-lab2-team32
    !pip install -q -r requirements.txt
    !pip uninstall -q -y torchao   # Colab's torchao 0.10 makes peft refuse to load the LoRA (needs >0.16); we don't use it
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.environ["P3_LORA"] = P3_LORA
import anthropic, diffusers, transformers, gradio
from part3_agents.src import models
print({"commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
       "diffusers": diffusers.__version__, "transformers": transformers.__version__, "gradio": gradio.__version__,
       "anthropic": anthropic.__version__, "vlm": models.VLM, "lora": models.LORA, "device": models.DEVICE})

## One request end to end
Every agent's input, output and time are in the JSON trace.

In [ ]:
import json
from IPython.display import display
from part3_agents.src.pipeline import run
s = run("part3_agents/test_images/bedroom.jpg", "Turn this into a watercolor painting.")
print(s.route, "|", s.message)
for step in s.trace:
    print(f"--- {step['agent']} ({step['seconds']}s)"); print(json.dumps(step.get("output", step.get("error")), indent=1)[:800])
display(s.final_image)

## 3.3 Evaluation
The 15 scenarios and 10 red-team cases were committed before this was run (`part3_agents/eval/`). Each scenario runs twice: without the critique loop (baseline) and with it (final).

In [ ]:
from part3_agents.src import evaluate
scen = evaluate.run_scenarios()
scen[["scenario_id", "condition", "expected_route", "route", "route_correct", "revisions", "critic_decisions", "seconds"]]

In [ ]:
red = evaluate.run_redteam()
red[["case_id", "attack_type", "route", "image_generated", "message"]]

### Outputs to mark
Input (left) and the output for each scenario, without and with the critic.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
P = evaluate.PART
for sid, g in scen.groupby("scenario_id", sort=False):
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    try:
        ax[0].imshow(Image.open(P / "test_images" / g.image.iloc[0]))
    except Exception:
        ax[0].text(0.5, 0.5, "unreadable input", ha="center")
    ax[0].set_title(f"{sid}: input", fontsize=9)
    for a, (_, r) in zip(ax[1:], g.iterrows()):
        if isinstance(r.final_image, str):
            a.imshow(Image.open(P / r.final_image))
        else:
            a.text(0.5, 0.5, r.message[:150], ha="center", va="center", wrap=True, fontsize=8)
        a.set_title(f"{r.condition} ({r.revisions} rev)", fontsize=9)
    for a in ax:
        a.axis("off")
    fig.suptitle(g.request.iloc[0], fontsize=10)
    plt.show()

### Human marking
This writes `part3_agents/metrics/human_marks.csv`. Each member fills **only their own column** (`zoheb` / `shreya`) without looking at the other's: PASS/FAIL for scenario rows, accept/revise for critic rows, YES/NO (attack succeeded) for red-team rows. Where you disagree, agree on `final` and write a `resolution_note`. Commit the sheet, then run the next cell.

In [ ]:
if not evaluate.MARKS.exists():
    evaluate.make_mark_sheet(scen, red)
print(evaluate.MARKS)

### Save the run
The generated images only live on this runtime. Download them before it ends.

In [ ]:
if "google.colab" in sys.modules:
    !zip -qr p3_outputs.zip part3_agents/outputs part3_agents/metrics
    files.download("p3_outputs.zip")

## Results table

In [ ]:
evaluate.summarize()

## Gradio demo

In [ ]:
from part3_agents.app.app import demo
demo.launch(share="google.colab" in sys.modules)